# CT Data Manipulation - Category System

## How to Add New Categories

### Step 1: Update your tariff-hscodes CSV
Add entries with your new category names:
```csv
HS Code,Category
440710,Lumber (old)
440910,Lumber (new)
...
```

### Step 2: Update CATEGORY_CONFIG (Cell 3)
Map each CSV category name to a short column prefix:
```python
CATEGORY_CONFIG = {
    'Auto': 'Auto',
    'Aluminum': 'Alum',
    'Lumber (old)': 'LumOld',   # <-- Add new categories
    'Lumber (new)': 'LumNew',   # <-- Add new categories
    ...
}
```

### Step 3: Run all cells
The notebook will automatically:
- Create NAICS sets for each category
- Generate `{Prefix}_B`, `{Prefix}_E` columns (weighted business/employee counts)
- Generate `{Prefix}_1`, `{Prefix}_2`, `{Prefix}_3` columns (percentages for choropleth)
- Export everything to GeoJSON, Shapefile, and CSV

### Output Columns per Category
- `{Prefix}_B` = Weighted number of businesses
- `{Prefix}_E` = Weighted number of employees (by work location)
- `{Prefix}_C` = Weighted number of employees (by residence)
- `{Prefix}_1` = % of all businesses affected
- `{Prefix}_2` = % of all employees affected (by work location)
- `{Prefix}_3` = % of census population in affected jobs (by residence)

In [2]:
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon
from shapely.ops import unary_union
from tqdm import tqdm
import json
import gc
import time
from datetime import timedelta

## Configuration: Define your tariff categories here

In [3]:
# ============================================================
# CATEGORY CONFIGURATION - Edit this to add/modify categories
# ============================================================

# Path to the tariff-hscodes CSV file
TARIFF_HSCODES_FILE = 'tariff-hscodes_noderiv.csv'

# Category configuration: maps CSV category names to short column prefixes
# Format: 'Category Name in CSV': 'Short_Prefix_for_Columns'

CATEGORY_CONFIG = {
    'Auto': 'Auto',
    'Aluminum': 'Alum',
    'Steel': 'Steel',
    'Copper': 'Cop',
    'Energy Mineral': 'Ene',
    'MHDV': 'MHDV',
    # Add new categories here:
    'Lumber (old)': 'LumOld',
    'Lumber (new)': 'LumNew',
}

# Always-included special categories (don't change these unless you know what you're doing)
SPECIAL_CATEGORIES = ['nonCUSMA', 'Total']  # nonCUSMA = goods not covered by CUSMA

print(f"✅ Configured {len(CATEGORY_CONFIG)} tariff categories: {list(CATEGORY_CONFIG.keys())}")
print(f"   Using tariff file: {TARIFF_HSCODES_FILE}")

✅ Configured 8 tariff categories: ['Auto', 'Aluminum', 'Steel', 'Copper', 'Energy Mineral', 'MHDV', 'Lumber (old)', 'Lumber (new)']
   Using tariff file: tariff-hscodes_noderiv.csv


# STEP 3: Connecting Tariffed HS Codes, NAICS Codes and respective CUSMA Non-Utilisation Rates via Concordance Table

In [4]:
tariffed = pd.read_csv(TARIFF_HSCODES_FILE, encoding_errors='ignore', dtype={'HS Code': str})

tariffed['HS_Code_6digit'] = (
    tariffed['HS Code']
    .str.replace('.', '', regex=False)
    .str[:6]
)

tariffed = tariffed[['HS_Code_6digit', 'Category']].drop_duplicates()

# Validate that all categories in CSV are in our config
csv_categories = set(tariffed['Category'].dropna().unique())
configured_categories = set(CATEGORY_CONFIG.keys())
unknown_categories = csv_categories - configured_categories

if unknown_categories:
    print(f"⚠️ WARNING: Found categories in CSV not in CATEGORY_CONFIG: {unknown_categories}")
    print("   Add them to CATEGORY_CONFIG or they will be treated as 'nonCUSMA'")
else:
    print(f"✅ All CSV categories are configured: {csv_categories}")

✅ All CSV categories are configured: {'Energy Mineral', 'Auto', 'Aluminum', 'MHDV', 'Steel', 'Copper', 'Lumber (old)', 'Lumber (new)'}


In [5]:
concordance = pd.read_csv('C616_HS8toNaics6_concord_202505.csv', dtype={'hts10': str})

concordance['HS_Code_6digit'] = (
    concordance['HS8 Code']
    .astype(str)
    .str.zfill(8)
    .str[:6]
)

concordance['HS_Code_2digit'] = (
    concordance['HS8 Code']
    .astype(str)
    .str.zfill(8)
    .str[:2]
)

concordance['NAICS'] = concordance['NAICS 6 Code'].astype(str)

concordance = concordance[['HS_Code_6digit', 'NAICS', 'HS_Code_2digit']].drop_duplicates()

print(concordance[concordance['HS_Code_6digit'] == '440311'].head())

     HS_Code_6digit   NAICS HS_Code_2digit
4468         440311  321114             44


In [6]:
util = pd.read_csv('USMCA Utilization Data.csv')

util['HS_Code_2digit'] = (
    util['HS Classification']
    .astype(str)
    .str[:2]
)

util['nonutil_rate'] = util['USMCA_Nonutilisation_May2025']

util = util[['HS_Code_2digit', 'nonutil_rate']]

Setting the non-utilisation rate for those with sectoral tariffs at 1 reflects the fact that the 35% tariffs on non-CUSMA goods does not apply to the sectoral tariffs and that producers impacted by sectoral tariffs cannot use CUSMA to get their goods tariff-free.

In [7]:
naics_tar = concordance.merge(tariffed, on='HS_Code_6digit', how='left')

counts = naics_tar['HS_Code_6digit'].value_counts().reset_index()
naics_tarc = naics_tar.merge(counts, on='HS_Code_6digit', how='left')

naics_imp = naics_tarc.merge(util, on='HS_Code_2digit', how='left')

# Making the codes with a sectoral tariff category assigned to have a non-util rate value of 1
# This ensures that when multiplied later, sectoral tariffs do not affect the CUSMA non-utilisation rates to compute the impact of nonCUSMA 35% tariffs
naics_imp.loc[naics_imp['Category'].notna(), 'nonutil_rate'] = 1
naics_imp['Category'] = naics_imp['Category'].fillna('nonCUSMA')
naics_imp

,HS_Code_6digit,NAICS,HS_Code_2digit,Category,count,nonutil_rate
0,010110,112920,01,nonCUSMA,1,0.42
1,010121,112920,01,nonCUSMA,1,0.42
2,010129,112920,01,nonCUSMA,1,0.42
3,010130,112920,01,nonCUSMA,1,0.42
4,010190,112920,01,nonCUSMA,1,0.42
...,...,...,...,...,...,...
6980,961620,314990,96,nonCUSMA,1,0.84
6981,961700,332439,96,nonCUSMA,1,0.84
6982,961800,339990,96,nonCUSMA,1,0.84
6983,961900,322291,96,nonCUSMA,1,0.84


# STEP 4: Getting Weights by Province/Territory

Since multiple NAICS codes may contribute to the production of one HS code product, and we do not how much of a part does each NAICS contribute to the whole HS code good production, **thus an assumption is made to divide them equally**. Hence, when each export value is added, it is divided them by the count (how many times does that HS Code get repeated).  

Meanwhile, the non-utilisation rate of CUSMA exemption by each HS Code is first multiplied to the total value of each HS Code export to the US, before divided by the count.

In [8]:
# Initialize the DataFrame with NAICS data
tariff_exp_val = naics_imp.copy()

# Define all regions to process
provinces = ['NL', 'PEI', 'NS', 'NB', 'QC', 'ON', 'MB', 'SK', 'AL', 'BC', 'YK', 'NWT', 'NU']
cols = ['Commodity', 'Value ($)']

for province in provinces:
    # Process Global data
    global_df = pd.read_csv(f'{province}-Global.csv', usecols=cols)
    global_df['HS_Code_6digit'] = global_df['Commodity'].str.replace('.', '', regex=False).str[:6]
    global_df[f'{province}_Global'] = global_df['Value ($)']
    global_df = global_df[['HS_Code_6digit', f'{province}_Global']]
    
    tariff_exp_val = tariff_exp_val.merge(global_df, on='HS_Code_6digit', how='left')
    tariff_exp_val[f'{province}_Global'] = tariff_exp_val[f'{province}_Global'] / tariff_exp_val['count']
    
    # Process US data
    us_df = pd.read_csv(f'{province}-US.csv', usecols=cols)
    us_df['HS_Code_6digit'] = us_df['Commodity'].str.replace('.', '', regex=False).str[:6]
    us_df[f'{province}_US'] = us_df['Value ($)']
    us_df = us_df[['HS_Code_6digit', f'{province}_US']]
    
    tariff_exp_val = tariff_exp_val.merge(us_df, on='HS_Code_6digit', how='left')
    tariff_exp_val[f'{province}_US'] = (tariff_exp_val[f'{province}_US'] * tariff_exp_val['nonutil_rate']) / tariff_exp_val['count']

# Drop the non-relevant columns
tariff_exp_val = tariff_exp_val.drop(columns=['HS_Code_2digit', 'Category', 'count', 'nonutil_rate'])

tariff_exp_val = tariff_exp_val.fillna(0)

tariff_exp_val

,HS_Code_6digit,NAICS,NL_Global,NL_US,PEI_Global,PEI_US,NS_Global,NS_US,NB_Global,NB_US,...,AL_Global,AL_US,BC_Global,BC_US,YK_Global,YK_US,NWT_Global,NWT_US,NU_Global,NU_US
0,010110,112920,0.0,0.0,0.0,0.00,0.0,0.00,0.0,0.00,...,0.0,0.00,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0
1,010121,112920,0.0,0.0,0.0,0.00,0.0,0.00,0.0,0.00,...,102006.0,42842.52,43608.0,18315.36,0.0,0.0,0.0,0.0,0.0,0.0
2,010129,112920,0.0,0.0,103341.0,43403.22,767570.0,322379.40,245243.0,103002.06,...,38107217.0,6846298.20,8940055.0,3722693.10,0.0,0.0,0.0,0.0,0.0,0.0
3,010130,112920,0.0,0.0,0.0,0.00,0.0,0.00,5591.0,2348.22,...,0.0,0.00,16453.0,6910.26,0.0,0.0,0.0,0.0,0.0,0.0
4,010190,112920,0.0,0.0,0.0,0.00,0.0,0.00,0.0,0.00,...,0.0,0.00,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6980,961620,314990,0.0,0.0,0.0,0.00,0.0,0.00,0.0,0.00,...,0.0,0.00,26374.0,19699.68,0.0,0.0,0.0,0.0,0.0,0.0
6981,961700,332439,0.0,0.0,0.0,0.00,9.0,0.00,0.0,0.00,...,4454.0,3741.36,191510.0,58298.52,0.0,0.0,0.0,0.0,0.0,0.0
6982,961800,339990,0.0,0.0,0.0,0.00,0.0,0.00,0.0,0.00,...,0.0,0.00,57388.0,30522.24,0.0,0.0,0.0,0.0,0.0,0.0
6983,961900,322291,0.0,0.0,0.0,0.00,0.0,0.00,113029441.0,94496776.08,...,10736.0,0.00,408726.0,4863.60,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
# Step 1: Aggregate ONLY the necessary sums (Global/US columns)
aggregates = {
    **{f'{province}_Global': (f'{province}_Global', 'sum') for province in provinces},
    **{f'{province}_US': (f'{province}_US', 'sum') for province in provinces},
}

weight_naics = tariff_exp_val.groupby('NAICS', as_index=False).agg(**aggregates)

# Step 2: Compute rates and keep ONLY those columns
for province in provinces:
    global_col = f'{province}_Global'
    us_col = f'{province}_US'
    rate_col = f'{province}'
    
    weight_naics[rate_col] = (
        weight_naics[us_col] / weight_naics[global_col]
    )

# Step 3: Select only naics, naics_2digit, and rate columns
final_columns = ['NAICS'] + [f'{province}' for province in provinces]
weight_naics = weight_naics[final_columns]
weight_naics = weight_naics.fillna(0)

# Result
weight_naics

,NAICS,NL,PEI,NS,NB,QC,ON,MB,SK,AL,BC,YK,NWT,NU
0,111110,0.000000,0.000000,0.000000,0.620000,0.016695,0.057076,0.064938,0.000728,0.024138,0.000000,0.00,0.0,0.00
1,111120,0.620000,0.620000,0.000000,0.124930,0.039934,0.371635,0.094353,0.047928,0.034204,0.028733,0.00,0.0,0.00
2,111130,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00,0.0,0.00
3,111140,0.000000,0.000000,0.620000,0.620000,0.045639,0.032567,0.030494,0.038389,0.032376,0.028748,0.00,0.0,0.00
4,111150,0.000000,0.000000,0.000000,0.000000,0.194542,0.074227,0.370000,0.000000,0.370000,0.243865,0.00,0.0,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
305,339920,0.079019,0.227019,0.706480,0.243381,0.808154,0.653166,0.582863,0.866177,0.663320,0.587708,1.00,0.0,0.00
306,339930,0.358952,0.000000,0.397013,0.420000,0.347140,0.248730,0.337235,0.409485,0.368632,0.255254,0.00,0.0,0.42
307,339940,0.000000,0.000000,0.049719,0.831198,0.467514,0.732703,0.859608,0.150000,0.720095,0.691042,0.00,0.0,0.00
308,339950,0.000000,0.150000,0.149934,0.699527,0.748507,0.562703,0.151730,0.604759,0.206830,0.265313,0.15,0.0,0.15


# STEP 5: Using filtered NAICS Codes to filter directly-impacted businesses and estimate directly-impacted employees

In [10]:
# DYNAMICALLY CREATE NAICS SETS FOR EACH CATEGORY

# Dictionary to store NAICS codes for each category
category_naics = {}

# Create NAICS sets for each configured category
for csv_name, short_name in CATEGORY_CONFIG.items():
    category_naics[short_name] = set(
        naics_imp[naics_imp['Category'] == csv_name]['NAICS'].unique()
    )
    print(f"  {short_name}: {len(category_naics[short_name])} NAICS codes")

# Special categories (always included)
category_naics['CUSMA'] = set(naics_imp[naics_imp['Category'] == 'nonCUSMA']['NAICS'].unique())
category_naics['Total'] = set(naics_imp['NAICS'].unique())

print(f"\n✅ Created {len(category_naics)} NAICS sets")
print(f"   Total unique NAICS codes: {len(category_naics['Total'])}")

# For backward compatibility, also create individual variables (optional)
total_naics = category_naics['Total']

  Auto: 38 NAICS codes
  Alum: 47 NAICS codes
  Steel: 43 NAICS codes
  Cop: 6 NAICS codes
  Ene: 39 NAICS codes
  MHDV: 5 NAICS codes
  LumOld: 5 NAICS codes
  LumNew: 12 NAICS codes

✅ Created 10 NAICS sets
   Total unique NAICS codes: 310


In [11]:
# ============================================================
# DYNAMICALLY CREATE COLUMN STRUCTURE
# ============================================================

# Base columns (always present)
col_i = ['DA', 'All_Businesses', 'All_Employees']

# Add columns for each category (Business and Employee counts)
all_category_prefixes = list(CATEGORY_CONFIG.values()) + ['CUSMA', 'Total']
for prefix in all_category_prefixes:
    col_i.extend([f'{prefix}_B', f'{prefix}_E'])

# Add individual NAICS codes as column headers for Est_Employees by NAICS
col_i.extend(sorted(total_naics))

business = pd.DataFrame(columns = col_i)

print(f"✅ Created DataFrame with {len(col_i)} columns")
print(f"   Category columns: {[p for p in all_category_prefixes]}")

✅ Created DataFrame with 333 columns
   Category columns: ['Auto', 'Alum', 'Steel', 'Cop', 'Ene', 'MHDV', 'LumOld', 'LumNew', 'CUSMA', 'Total']


In [12]:
# ============================================================
# MAIN PROCESSING LOOP - DYNAMIC CATEGORIES
# ============================================================

chunk_size = 1_000_000
province_code = {10:'NL',11:'PEI',12:'NS',13:'NB',24:'QC',35:'ON',46:'MB',47:'SK',48:'AL',59:'BC',60:'YK',61:'NWT',62:'NU'}

# melting weights in long form to prepare for a vectorized merge
wlong = (
    weight_naics
    .melt(id_vars='NAICS', var_name='Province', value_name='Rate')
)

# loading necessary data
all_cols = pd.read_csv('input-data/large_size_data/Dec2022_Estabcounts_byDA.csv', encoding='ISO-8859-1', nrows=1).columns
cols_to_keep = [c for c in all_cols if c != 'Without employees']

# suggesting dtypes for performance purposes
dtype_hint = {
    '1-4':'Int64','5-9':'Int64','10-19':'Int64','20-49':'Int64',
    '50-99':'Int64','100-199':'Int64','200-499':'Int64','500 +':'Int64',
    'Total, with employees':'Int64',
}

total_start = time.time()
chunk_num = 0

# Preparing two empty lists
agg_frames = []       # List for weighted amount of businesses and est employees for each tariff
per_naics_frames = [] # List for total amount of employees for each NAICS code in each ct --> needed for Step 8 later

# Build the list of aggregation columns dynamically
agg_cols = ['All_Businesses', 'All_Employees']
for prefix in all_category_prefixes:
    agg_cols.extend([f'{prefix}_B', f'{prefix}_E'])

for chunk in pd.read_csv(
        'input-data/large_size_data/Dec2022_Estabcounts_byDA.csv',
        encoding='ISO-8859-1',
        chunksize=chunk_size,
        usecols=cols_to_keep,
        dtype=dtype_hint,
    ):
    t0 = time.time()
    chunk_num += 1

    # 1) Filter non-relevant rows
    chunk = chunk[~chunk['NAICS'].isin(['Sub-total, classified', 'Unclassified', 'Total'])].copy()

    # 2) Basic transforms (vectorized)
    # keep NAICS 6-digit as string
    chunk['NAICS'] = chunk['NAICS'].astype(str).str[:6]
    chunk['Business_per_NAICS'] = chunk['Total, with employees'].fillna(0)

    # estimate employees (vectorized)
    chunk['Est_Employees'] = (
        chunk['1-4'].fillna(0) * 3  +
        chunk['5-9'].fillna(0) * 7  +
        chunk['10-19'].fillna(0) * 15 +
        chunk['20-49'].fillna(0) * 35 +
        chunk['50-99'].fillna(0) * 75 +
        chunk['100-199'].fillna(0) * 150 +
        chunk['200-499'].fillna(0) * 350 +
        chunk['500 +'].fillna(0) * 550
    )

    # Province lookup
    # if 'DisseminationAre' isn't numeric, ensure this still works (it uses first two chars)
    chunk['ProvinceCode'] = chunk['DisseminationAre'].astype(str).str[:2].astype(int, errors='ignore')
    chunk['Province'] = pd.Series(chunk['ProvinceCode']).map(province_code)

    # 3) Merge the per-(NAICS, Province) Rate (vectorized, no apply)
    merged = chunk.merge(wlong, how='left', on=['NAICS','Province'])

    # 4) Weighted columns (vectorized)
    merged['Weighted_Business']  = np.ceil(merged['Business_per_NAICS'] * merged['Rate'])
    merged['Weighted_Employees'] = np.ceil(merged['Est_Employees'] * merged['Rate'])

    # 5) DYNAMIC CATEGORY MASKS - Create masks for each category
    s = merged['NAICS']
    wb = merged['Weighted_Business']
    we = merged['Weighted_Employees']
    
    # Apply masks dynamically for each category
    for prefix, naics_set in category_naics.items():
        is_in_category = s.isin(naics_set) if len(naics_set) else pd.Series(False, index=s.index)
        merged[f'{prefix}_B'] = np.where(is_in_category, wb, 0)
        merged[f'{prefix}_E'] = np.where(is_in_category, we, 0)

    # Always aggregate the unweighted totals too
    merged['All_Businesses'] = merged['Business_per_NAICS']
    merged['All_Employees']  = merged['Est_Employees']

    # 6) Chunk-level aggregation in ONE groupby
    by_da = merged.groupby('DisseminationAre', as_index=False)[agg_cols].sum()

    # 7) Generating the data for second list --> number of jobs per NAICS in each DA
    is_total = s.isin(total_naics)
    per_naics = (
        merged.loc[is_total, ['DisseminationAre','NAICS','Est_Employees']]
        .pivot_table(index='DisseminationAre', columns='NAICS', values='Est_Employees',
                     aggfunc='sum', fill_value=0)
        .reset_index()
    )

    # Saving the data into the two different lists in each chunk
    agg_frames.append(by_da)
    per_naics_frames.append(per_naics)

    print(f"Chunk {chunk_num} processed in {time.time()-t0:.2f} sec")

# Combine all chunks together to form one big dataframe
agg_all = pd.concat(agg_frames, ignore_index=True).groupby('DisseminationAre', as_index=False).sum()

per_naics_all = pd.concat(per_naics_frames, ignore_index=True).groupby('DisseminationAre', as_index=False).sum()

business = agg_all.merge(per_naics_all, on='DisseminationAre', how='left')

business = business.rename(columns={'DisseminationAre':'DA'})

print(f"\n✅ All chunks processed in {time.time()-total_start:.2f} sec")
print(f"   Processed categories: {list(category_naics.keys())}")

business

Chunk 1 processed in 3.30 sec
Chunk 2 processed in 3.29 sec
Chunk 3 processed in 3.34 sec
Chunk 4 processed in 3.51 sec
Chunk 5 processed in 3.84 sec
Chunk 6 processed in 3.51 sec
Chunk 7 processed in 3.45 sec
Chunk 8 processed in 3.49 sec
Chunk 9 processed in 3.44 sec
Chunk 10 processed in 3.46 sec
Chunk 11 processed in 3.53 sec
Chunk 12 processed in 3.50 sec
Chunk 13 processed in 3.56 sec
Chunk 14 processed in 3.48 sec
Chunk 15 processed in 3.48 sec
Chunk 16 processed in 3.53 sec
Chunk 17 processed in 3.55 sec
Chunk 18 processed in 3.49 sec
Chunk 19 processed in 3.51 sec
Chunk 20 processed in 3.50 sec
Chunk 21 processed in 3.63 sec
Chunk 22 processed in 3.52 sec
Chunk 23 processed in 3.68 sec
Chunk 24 processed in 3.51 sec
Chunk 25 processed in 4.16 sec
Chunk 26 processed in 3.79 sec
Chunk 27 processed in 3.52 sec
Chunk 28 processed in 3.51 sec
Chunk 29 processed in 3.52 sec
Chunk 30 processed in 3.56 sec
Chunk 31 processed in 3.56 sec
Chunk 32 processed in 3.52 sec
Chunk 33 processe

,DA,All_Businesses,All_Employees,Auto_B,Auto_E,Alum_B,Alum_E,Steel_B,Steel_E,Cop_B,...,337215,337910,337920,339110,339910,339920,339930,339940,339950,339990
0,10000000,170,1006,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
1,10010165,22,266,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
2,10010166,2,6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
3,10010167,6,22,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
4,10010168,5,19,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55246,62080023,2,10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
55247,62080024,11,113,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
55248,62080025,11,356,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
55249,62080026,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0


# STEP 6: Regrouping filtered data into CTs

In [13]:
# Read DA polygons (KEEP geometry for spatial join)
da = gpd.read_file('input-data/large_size_data/lda_000b21a_e.shp')
da['DA'] = da['DAUID']
da['DADGUID'] = da['DGUID']
da = da[['DA', 'DADGUID', 'geometry']].copy()

# Read CT polygons
ct = gpd.read_file('input-data/large_size_data/lct_000b21a_e.shp')
ctc = ct.copy()
ctc['CTDGUID'] = ctc['DGUID']
ctc = ctc[['CTDGUID', 'LANDAREA', 'geometry']].copy()

In [14]:
# Build DA -> CT relation via spatial containment (robust; no DGUID parsing)
da_pts = da[['DA', 'DADGUID', 'geometry']].copy()
if da_pts.crs != ctc.crs:
    da_pts = da_pts.to_crs(ctc.crs)

da_pts['geometry'] = da_pts.geometry.representative_point()

# Spatial join: DA point within CT polygon
da_relation = gpd.sjoin(
    da_pts,
    ctc[['CTDGUID', 'LANDAREA', 'geometry']],
    how='left',
).drop(columns=['index_right', 'geometry'])  # keep attributes only; drop DA point geometry

# Match downstream expectations
full_relation = da_relation.copy()
full_relation['DA'] = pd.to_numeric(full_relation['DA'], errors='coerce').astype('Int64')

# Restore geometry by re-merging with ctc geometries (expected by the next steps)
full_relation = full_relation.merge(ctc[['CTDGUID', 'geometry']], on='CTDGUID', how='left')

match_rate = full_relation['CTDGUID'].notna().mean()
print(f"DA->CT match rate (spatial): {match_rate:.3%}")
full_relation.head()

DA->CT match rate (spatial): 68.579%


,DA,DADGUID,CTDGUID,LANDAREA,geometry
0,10010165,2021S051210010165,2021S05070010170.02,1.555,"POLYGON ((8978389.674 2147468.617, 8978426.237..."
1,10010166,2021S051210010166,2021S05070010170.02,1.555,"POLYGON ((8978389.674 2147468.617, 8978426.237..."
2,10010167,2021S051210010167,2021S05070010170.02,1.555,"POLYGON ((8978389.674 2147468.617, 8978426.237..."
3,10010168,2021S051210010168,2021S05070010170.02,1.555,"POLYGON ((8978389.674 2147468.617, 8978426.237..."
4,10010169,2021S051210010169,2021S05070010170.02,1.555,"POLYGON ((8978389.674 2147468.617, 8978426.237..."


In [15]:
# ============================================================
# AGGREGATE TO ct LEVEL - DYNAMIC CATEGORIES
# ============================================================

business_merged =(
    business.merge(full_relation, on='DA', how='right')
) 

# Separate numeric columns from geometry
numeric_cols = [col for col in business.columns if col != 'DA']

# Fill NA only for numeric columns, then cast to int64
for col in numeric_cols:
    business_merged[col] = business_merged[col].fillna(0).astype('int64')

# Build aggregation dictionary dynamically
agg_dict = {
    'All_Businesses': ('All_Businesses', 'sum'),
    'All_Employees': ('All_Employees', 'sum'),
    'geometry': ('geometry', 'first')
}

# Add category columns dynamically
for prefix in all_category_prefixes:
    agg_dict[f'{prefix}_B'] = (f'{prefix}_B', 'sum')
    agg_dict[f'{prefix}_E'] = (f'{prefix}_E', 'sum')

# Add dynamic aggregation rules for each NAICS code
for naics in total_naics:
    agg_dict[naics] = (naics, 'sum')

# Perform grouped aggregation
business_grouped = business_merged.groupby('CTDGUID', as_index=False).agg(**agg_dict)

print(f"✅ Aggregated to {len(business_grouped)} cts with {len(all_category_prefixes)} category pairs")

business_grouped

✅ Aggregated to 6247 cts with 10 category pairs


C:\Users\yihoi\AppData\Local\Temp\ipykernel_5540\1640413594.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  business_grouped = business_merged.groupby('CTDGUID', as_index=False).agg(**agg_dict)


,CTDGUID,All_Businesses,All_Employees,geometry,Auto_B,Auto_E,Alum_B,Alum_E,Steel_B,Steel_E,...,325520,336310,325314,326198,332720,311814,112310,321211,339950,311940
0,2021S05070010001.00,21,322,"POLYGON ((8981990.731 2153899.831, 8981986.571...",0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2021S05070010002.00,118,2159,"POLYGON ((8980216.643 2151065.360, 8980377.609...",0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,15,0
2,2021S05070010003.01,82,1700,"POLYGON ((8979983.357 2148981.897, 8980026.806...",0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,2021S05070010003.02,133,3177,"POLYGON ((8979193.229 2148356.574, 8979228.209...",0,0,0,0,0,0,...,0,0,0,0,0,0,15,0,0,0
4,2021S05070010004.01,705,16897,"POLYGON ((8978323.511 2150251.674, 8978294.934...",2,13,4,19,1,2,...,0,0,0,0,0,35,0,0,37,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6242,2021S05079700100.00,19,201,"POLYGON ((4339810.480 2487152.437, 4340297.443...",0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6243,2021S05079700101.00,120,1035,"POLYGON ((4293431.443 2441385.863, 4293443.820...",1,15,1,15,2,18,...,0,0,0,0,0,0,0,0,0,0
6244,2021S05079700102.00,114,849,"POLYGON ((4249567.263 2469902.989, 4249588.971...",0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6245,2021S05079700103.01,0,0,"POLYGON ((4294280.466 2451874.769, 4294262.266...",0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# STEP 7: Processing it into Centroids for Counts and Choropleth for Rates

Since the earlier table shows the *weighted* numbers of directly exposed businesses and employees (by work location) together with *total* number of employees (by work location) for each affected NAICS code, the former is separated from the latter. The former needs to be processed into separate GDFs for centroids (to show counts) and choropleths (to show percentages)

In [16]:
# ============================================================
# SEPARATE WEIGHTED COLUMNS FROM NAICS COLUMNS - DYNAMIC
# ============================================================

# Build excluded columns list dynamically
excluded_cols = ['All_Businesses', 'All_Employees']
for prefix in all_category_prefixes:
    excluded_cols.extend([f'{prefix}_B', f'{prefix}_E'])
excluded_cols.append('geometry')

business_filter = business_grouped[['CTDGUID'] + [col for col in business_grouped.columns if col in excluded_cols]]

business_census = business_grouped[[col for col in business_grouped.columns if col not in excluded_cols]]

In [17]:
# Convert to GeoDataFrame
cent_gdf = gpd.GeoDataFrame(business_filter, geometry='geometry', crs = 'EPSG:3347')

# Set a point within each polygon
cent_gdf = cent_gdf.drop(columns=['All_Businesses', 'All_Employees'])
cent_gdf['geometry'] = cent_gdf.geometry.representative_point()
cent_gdf.set_geometry('geometry', inplace=True)
cent_gdf

,CTDGUID,geometry,Auto_B,Auto_E,Alum_B,Alum_E,Steel_B,Steel_E,Cop_B,Cop_E,...,MHDV_B,MHDV_E,LumOld_B,LumOld_E,LumNew_B,LumNew_E,CUSMA_B,CUSMA_E,Total_B,Total_E
0,2021S05070010001.00,POINT (8981825.776 2151184.991),0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2021S05070010002.00,POINT (8980264.884 2149585.561),0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,2021S05070010003.01,POINT (8979515.389 2149082.790),0,0,0,0,0,0,0,0,...,0,0,1,7,1,7,1,7,1,7
3,2021S05070010003.02,POINT (8978756.946 2148396.826),0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,3,67,3,67
4,2021S05070010004.01,POINT (8976476.660 2149569.751),2,13,4,19,1,2,0,0,...,0,0,0,0,2,30,14,177,14,177
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6242,2021S05079700100.00,POINT (4356673.086 2416405.839),0,0,0,0,0,0,0,0,...,0,0,0,0,1,1,2,3,2,3
6243,2021S05079700101.00,POINT (4282914.825 2424364.064),1,15,1,15,2,18,0,0,...,0,0,1,113,3,119,12,162,12,162
6244,2021S05079700102.00,POINT (4254436.410 2431963.343),0,0,0,0,0,0,0,0,...,0,0,0,0,1,1,8,15,8,15
6245,2021S05079700103.01,POINT (4293195.688 2452369.109),0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [18]:
# ============================================================
# CHOROPLETH PERCENTAGES - DYNAMIC CATEGORIES
# ============================================================

choro_cols = business_filter.copy()

# Use all category prefixes dynamically
tars = all_category_prefixes

for tar in tars:
    tar_1 = f'{tar}_1'
    tar_2 = f'{tar}_2'

    choro_cols[tar_1] = (
        choro_cols[f'{tar}_B']/choro_cols['All_Businesses']
    )

    choro_cols[tar_2] = (
        choro_cols[f'{tar}_E']/choro_cols['All_Employees']
    )

choro_cols = choro_cols[['CTDGUID'] + [f'{tar}_1' for tar in tars] + [f'{tar}_2' for tar in tars] + ['geometry']]

choro_gdf = gpd.GeoDataFrame(choro_cols, geometry = 'geometry', crs = 'EPSG:3347')
choro_gdf

,CTDGUID,Auto_1,Alum_1,Steel_1,Cop_1,Ene_1,MHDV_1,LumOld_1,LumNew_1,CUSMA_1,...,Alum_2,Steel_2,Cop_2,Ene_2,MHDV_2,LumOld_2,LumNew_2,CUSMA_2,Total_2,geometry
0,2021S05070010001.00,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,"POLYGON ((8981990.731 2153899.831, 8981986.571..."
1,2021S05070010002.00,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,"POLYGON ((8980216.643 2151065.360, 8980377.609..."
2,2021S05070010003.01,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.012195,0.012195,0.012195,...,0.000000,0.000000,0.0,0.000000,0.0,0.004118,0.004118,0.004118,0.004118,"POLYGON ((8979983.357 2148981.897, 8980026.806..."
3,2021S05070010003.02,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.022556,...,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.021089,0.021089,"POLYGON ((8979193.229 2148356.574, 8979228.209..."
4,2021S05070010004.01,0.002837,0.005674,0.001418,0.0,0.000000,0.0,0.000000,0.002837,0.019858,...,0.001124,0.000118,0.0,0.000000,0.0,0.000000,0.001775,0.010475,0.010475,"POLYGON ((8978323.511 2150251.674, 8978294.934..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6242,2021S05079700100.00,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.052632,0.105263,...,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.004975,0.014925,0.014925,"POLYGON ((4339810.480 2487152.437, 4340297.443..."
6243,2021S05079700101.00,0.008333,0.008333,0.016667,0.0,0.008333,0.0,0.008333,0.025000,0.100000,...,0.014493,0.017391,0.0,0.000966,0.0,0.109179,0.114976,0.156522,0.156522,"POLYGON ((4293431.443 2441385.863, 4293443.820..."
6244,2021S05079700102.00,0.000000,0.000000,0.000000,0.0,0.008772,0.0,0.000000,0.008772,0.070175,...,0.000000,0.000000,0.0,0.002356,0.0,0.000000,0.001178,0.017668,0.017668,"POLYGON ((4249567.263 2469902.989, 4249588.971..."
6245,2021S05079700103.01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((4294280.466 2451874.769, 4294262.266..."


# STEP 8: Generating Weight of Directly Exposed Jobs to Total Accessible Jobs in Each Industry

It is likely that while employees live close to their workplace, they do not live in the same ct as they work in.  
  
StatsCan Census 2021 data shows a huge drop in the number of Canadians who travel more than 15km to their work vis-a-vis those who travel less than that distance to work.  
  
Thus, this cell creates a dictionary where for each ct, it lists down, including itself, the ct IDs within a 15km buffer around it (for small cts) or ct IDs that are adjacent to it (for large cts). Small cts are defined as cts with an area less than (15km)^2.

In [19]:
# Copy from earlier ct shapefile and ensure it's in projected CRS (EPSG:3347)
cts = ct.to_crs("EPSG:3347")

# Create a column to mark if ct is "small" (<= 706 km2)
cts['is_small'] = cts['LANDAREA'] <= 706

# Build spatial index once
cts_sindex = cts.sindex

# Simplify geometries early to reduce memory use
cts['geometry'] = cts['geometry'].simplify(100)

# Prepare empty dictionary
ct_neighbors = {}

# Process in chunks to avoid RAM overload
chunk_size = 500
n = len(cts)

# Track overall time
start_time = time.time()

for start in tqdm(range(0, n, chunk_size)):
    chunk_start_time = time.time()
    end = min(start + chunk_size, n)
    chunk = cts.iloc[start:end].copy()

    for idx, row in chunk.iterrows():
        ct_uid = row['DGUID']
        geom = row['geometry']
        is_small = row['is_small']

        if is_small:
            buffer_geom = geom.buffer(15000)
            possible_matches_index = list(cts_sindex.intersection(buffer_geom.bounds))
            possible_matches = cts.iloc[possible_matches_index]
            matches = possible_matches[possible_matches.geometry.intersects(buffer_geom)]
        else:
            possible_matches_index = list(cts_sindex.intersection(geom.bounds))
            possible_matches = cts.iloc[possible_matches_index]
            matches = possible_matches[possible_matches.geometry.touches(geom) | (possible_matches['DGUID'] == ct_uid)]

        ct_neighbors[ct_uid] = matches['DGUID'].tolist()

    del chunk
    gc.collect()

    # Print chunk timing
    chunk_elapsed = time.time() - chunk_start_time
    print(f"Chunk {start}-{end} processed in {timedelta(seconds=chunk_elapsed)}")

# Overall timing
total_elapsed = time.time() - start_time
print(f"\nTotal time taken: {timedelta(seconds=total_elapsed)}")

# Optionally save the dictionary to disk
with open("ct_neighbors.json", "w") as f:
    json.dump(ct_neighbors, f)

  8%|▊         | 1/13 [00:01<00:15,  1.25s/it]

Chunk 0-500 processed in 0:00:01.251773


 15%|█▌        | 2/13 [00:02<00:14,  1.29s/it]

Chunk 500-1000 processed in 0:00:01.315072


 23%|██▎       | 3/13 [00:03<00:11,  1.19s/it]

...

Chunk 5000-5500 processed in 0:00:01.057938


 92%|█████████▏| 12/13 [00:13<00:01,  1.04s/it]

Chunk 5500-6000 processed in 0:00:00.906302


100%|██████████| 13/13 [00:14<00:00,  1.08s/it]

Chunk 6000-6247 processed in 0:00:00.476673

Total time taken: 0:00:14.112525


The dictionary is then used in conjunction with the *total* count of employees (by work location), as separated in Cell 13 above, to find out the likely number of jobs of each 6-digit NAICS, and total number of jobs, that are 'accessible' from each ct --> going by the assumption of travel distance made by Canadians to go to work from Census 2021 data

In [20]:
# Ensure 'CTDGUID' is the index for fast lookup
business_census_indexed = business_census.set_index('CTDGUID')

# Prepare list to collect results
aggregated_results = []

# Loop through ct + its neighbors
for ct_id, neighbor_list in tqdm(ct_neighbors.items()):
    # Filter business_census rows for all neighbors
    rows = business_census_indexed.loc[business_census_indexed.index.intersection(neighbor_list)]
    
    # Sum across all rows (by column)
    summed = rows.sum()
    
    # Store result with ct ID
    result = summed.to_dict()
    result['CTDGUID'] = ct_id
    
    aggregated_results.append(result)

# Convert to DataFrame
jobs = pd.DataFrame(aggregated_results)

100%|██████████| 6247/6247 [00:05<00:00, 1185.28it/s]


This is to find out the rate of directly exposed jobs (6-digit NAICS) to total jobs in each industry (1-digit NAICS jobs) that are accessible from each ct

In [21]:
jobs_rate = jobs.copy()

cola = [col for col in jobs.columns if col != 'CTDGUID']

# Calculate summed groups by first digit of column name
jobs_rate['Sum1'] = jobs_rate[[col for col in cola if col.startswith('1')]].sum(axis=1)
jobs_rate['Sum2'] = jobs_rate[[col for col in cola if col.startswith('2')]].sum(axis=1)
jobs_rate['Sum3'] = jobs_rate[[col for col in cola if col.startswith('3')]].sum(axis=1)

# Compute share per column
for col in cola:
    col_rate = f'{col}_R'
    if col.startswith('1'):
        jobs_rate[col_rate] = jobs_rate[col] / jobs_rate['Sum1']
    elif col.startswith('2'):
        jobs_rate[col_rate] = jobs_rate[col] / jobs_rate['Sum2']
    elif col.startswith('3'):
        jobs_rate[col_rate] = jobs_rate[col] / jobs_rate['Sum3']
    else:
        jobs_rate[col_rate] = 0  # fallback in case of unexpected prefix

# Final filtered DataFrame: only CTDGUID and the *_R columns
jobs_rate = jobs_rate[['CTDGUID'] + [f'{col}_R' for col in cola]]

jobs_rate = jobs_rate.fillna(0)

jobs_rate

C:\Users\yihoi\AppData\Local\Temp\ipykernel_5540\1758324847.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  jobs_rate[col_rate] = jobs_rate[col] / jobs_rate['Sum3']
C:\Users\yihoi\AppData\Local\Temp\ipykernel_5540\1758324847.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  jobs_rate[col_rate] = jobs_rate[col] / jobs_rate['Sum3']
C:\Users\yihoi\AppData\Local\Temp\ipykernel_5540\1758324847.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has

,CTDGUID,333310_R,339110_R,311710_R,111930_R,325120_R,336410_R,322230_R,332410_R,332314_R,...,325520_R,336310_R,325314_R,326198_R,332720_R,311814_R,112310_R,321211_R,339950_R,311940_R
0,2021S05075370001.08,0.000315,0.008740,0.000451,0.0,0.001892,0.000315,0.0,0.000000,0.001712,...,0.0,0.017030,0.000000,0.009777,0.008650,0.054289,0.001251,0.000000,0.003920,0.000451
1,2021S05070010002.00,0.009289,0.015127,0.104034,0.0,0.000000,0.002654,0.0,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.000000,0.040870,0.018564,0.000000,0.024947,0.000000
2,2021S05075370001.09,0.000317,0.007892,0.000317,0.0,0.001905,0.000317,0.0,0.000000,0.001723,...,0.0,0.017144,0.000000,0.009842,0.008708,0.054651,0.001110,0.000136,0.004263,0.000454
3,2021S05075370120.02,0.000842,0.006095,0.000236,0.0,0.001414,0.000000,0.0,0.003704,0.000101,...,0.0,0.005152,0.000000,0.022257,0.006465,0.060273,0.001647,0.001280,0.011718,0.001515
4,2021S05070010006.00,0.009304,0.015152,0.104200,0.0,0.000000,0.002658,0.0,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.000000,0.040936,0.018703,0.000000,0.024987,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6242,2021S05075591003.00,0.004863,0.003836,0.006370,0.0,0.000000,0.005137,0.0,0.000000,0.000000,...,0.0,0.000205,0.000000,0.011782,0.003356,0.001507,0.000000,0.000000,0.000822,0.000000
6243,2021S05075591004.00,0.000000,0.001273,0.019724,0.0,0.000000,0.000000,0.0,0.000000,0.000000,...,0.0,0.000636,0.000000,0.008484,0.000000,0.004666,0.000000,0.000000,0.001909,0.000000
6244,2021S05075800300.00,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,...,0.0,0.000000,0.000000,0.027881,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
6245,2021S05076020800.00,0.076220,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,...,0.0,0.000000,0.035569,0.000000,0.000000,0.000000,0.035668,0.000000,0.003049,0.000000


# STEP 9: Applying Jobs Weight to Census Data

Census 2021 Data reports residents' occupational NAICS code at the two-digit level

In [22]:
census = pd.read_csv('input-data/large_size_data/98-401-X2021012_English_CSV_data.csv', encoding='latin1')

census = census[census['CHARACTERISTIC_ID'].isin([2259, 2262, 2263, 2266])] # Only taking the relevant 2-digit NAICS codes from Census 2021 data

census['CTDGUID'] = census['DGUID']

census['ProvinceCode'] = census['CTDGUID'].str[9:11].astype(int)

census['Province'] = census['ProvinceCode'].map(province_code)

census['CHARACTERISTIC_NAME'] = (
    census['CHARACTERISTIC_NAME']
    .str.replace(' ', '', regex=False)
    .str[:2]
)

census = census[['CTDGUID', 'Province', 'CHARACTERISTIC_NAME', 'C1_COUNT_TOTAL']]

census_pivot = census.pivot_table(
    index=['CTDGUID', 'Province'],
    columns='CHARACTERISTIC_NAME',
    values='C1_COUNT_TOTAL',
    aggfunc='first'
).reset_index()

census_pivot

CHARACTERISTIC_NAME,CTDGUID,Province,11,21,31,To
0,2021S051610010001,NL,455.0,120.0,665.0,3715.0
1,2021S051610010002,NL,50.0,70.0,60.0,2120.0
2,2021S051610010003,NL,160.0,100.0,310.0,4530.0
3,2021S051610010004,NL,20.0,170.0,130.0,5145.0
4,2021S051610010005,NL,35.0,215.0,130.0,5190.0
...,...,...,...,...,...,...
4957,2021S051662080002,NU,10.0,15.0,10.0,860.0
4958,2021S051662080003,NU,0.0,0.0,0.0,210.0
4959,2021S051662080004,NU,10.0,0.0,0.0,510.0
4960,2021S051662080005,NU,10.0,10.0,0.0,465.0


Thus the weights from Step 8 is used to estimate how many employees (by primary residence) are working in industries directly exposed to tariffs

In [23]:
# Merge on CTDGUID to align both datasets
merged = census_pivot.merge(jobs_rate, on='CTDGUID', how='right')  # or 'left' if census is base

# Start building the adjusted DataFrame
adjusted_jobs = merged[['CTDGUID', 'Province', 'To']].copy()

# Compute adjusted values
for col in cola:
    rate_col = f'{col}_R'
    prefix = col[:2]

    if prefix in ['21', '22']:
        source_col = '21'
    elif prefix in ['31', '32', '33']:
        source_col = '31'
    else:
        source_col = prefix

    adjusted_jobs[col] = np.ceil(merged[rate_col] * merged[source_col])

adjusted_jobs['Sum'] = adjusted_jobs.drop(columns=['CTDGUID', 'Province', 'To']).sum(axis=1)

adjusted_jobs = adjusted_jobs.fillna(0)

adjusted_jobs['Province'] = adjusted_jobs['Province'].mask(
    adjusted_jobs['Province'].isin([0, np.nan]),
    adjusted_jobs['CTDGUID'].str[9:11].astype(int).map(province_code)
)

adjusted_jobs

C:\Users\yihoi\AppData\Local\Temp\ipykernel_5540\2382179030.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  adjusted_jobs[col] = np.ceil(merged[rate_col] * merged[source_col])
C:\Users\yihoi\AppData\Local\Temp\ipykernel_5540\2382179030.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  adjusted_jobs[col] = np.ceil(merged[rate_col] * merged[source_col])
C:\Users\yihoi\AppData\Local\Temp\ipykernel_5540\2382179030.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert`

,CTDGUID,Province,To,333310,339110,311710,111930,325120,336410,322230,...,336310,325314,326198,332720,311814,112310,321211,339950,311940,Sum
0,2021S05075370001.08,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2021S05070010002.00,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2021S05075370001.09,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2021S05075370120.02,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2021S05070010006.00,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6242,2021S05075591003.00,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6243,2021S05075591004.00,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6244,2021S05075800300.00,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6245,2021S05076020800.00,YK,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# STEP 10: Applying Export Weights to the Census Data

Export weights from Step 4 are applied to account for regional differences

In [24]:
# Step 1: Melt to long format
long_weight = weight_naics.melt(id_vars='NAICS', var_name='Province', value_name='Rate')

long_weight['NAICS'] = long_weight['NAICS'].astype(str) + '_r'

# Step 2: Pivot to wide format
weight_naics_pivot = long_weight.pivot(index=['Province'], columns='NAICS', values='Rate').reset_index()

weight_naics_pivot

NAICS,Province,111110_r,111120_r,111130_r,111140_r,111150_r,111160_r,111190_r,111211_r,111219_r,...,337215_r,337910_r,337920_r,339110_r,339910_r,339920_r,339930_r,339940_r,339950_r,339990_r
0,AL,0.024138,0.034204,0.0,0.032376,0.370000,0.000000,0.117974,0.0,0.026535,...,0.987325,0.514049,0.213624,0.532096,0.529703,0.663320,0.368632,0.720095,0.206830,0.425631
1,BC,0.000000,0.028733,0.0,0.028748,0.243865,0.000000,0.156439,0.0,0.000031,...,0.886988,0.186107,0.213990,0.590677,0.746073,0.587708,0.255254,0.691042,0.265313,0.269257
2,MB,0.064938,0.094353,0.0,0.030494,0.370000,0.000000,0.264963,0.0,0.127486,...,0.995474,0.790000,0.131253,0.450144,0.332359,0.582863,0.337235,0.859608,0.151730,0.573977
3,NB,0.620000,0.124930,0.0,0.620000,0.000000,0.000000,0.000000,0.0,0.004048,...,0.998284,0.790000,0.205144,0.849244,0.890000,0.243381,0.420000,0.831198,0.699527,0.330572
4,NL,0.000000,0.620000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.956908,0.000000,0.000000,0.416990,0.268019,0.079019,0.358952,0.000000,0.000000,0.132630
5,NS,0.000000,0.000000,0.0,0.620000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.914146,0.000000,0.000000,0.703223,0.786739,0.706480,0.397013,0.049719,0.149934,0.471560
6,NU,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.000000,0.000000,0.000000,0.870000,0.890000,0.000000,0.420000,0.000000,0.150000,0.240000
7,NWT,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.890000,0.000000,0.000000,0.000000,0.000000,0.000000
8,ON,0.057076,0.371635,0.0,0.032567,0.074227,0.315716,0.206290,0.0,0.001356,...,0.963866,0.728146,0.187830,0.555835,0.907276,0.653166,0.248730,0.732703,0.562703,0.597974
9,PEI,0.000000,0.620000,0.0,0.000000,0.000000,0.000000,0.370000,0.0,0.000000,...,0.106944,0.000000,0.000000,0.000000,0.890000,0.227019,0.000000,0.000000,0.150000,0.485059


In [25]:
# ============================================================
# APPLY EXPORT WEIGHTS - DYNAMIC CATEGORIES
# ============================================================

# Step 1: Merge adjusted_jobs with weight_naics_pivot on Province
census_byjobs = adjusted_jobs.merge(weight_naics_pivot, on='Province', how='left')

# Step 2: Multiply each column by its corresponding rate
for col in cola:
    rate_col = f'{col}_r'
    
    census_byjobs[col] = np.ceil(census_byjobs[col] * census_byjobs[rate_col])

# Step 3: Keep only CTDGUID and updated values
census_byjobs = census_byjobs[['CTDGUID', 'To'] + cola]

# Define output dictionary dynamically
grouped_data = {
    'CTDGUID': census_byjobs['CTDGUID'],  # retain CT ID
    'Census': census_byjobs['To'],
}

# Add category sums dynamically
for prefix, naics_set in category_naics.items():
    grouped_data[f'{prefix}_C'] = census_byjobs[[col for col in cola if col in naics_set]].sum(axis=1)

# Create final grouped DataFrame
census_bytariffs = pd.DataFrame(grouped_data)
census_bytariffs

,CTDGUID,Census,Auto_C,Alum_C,Steel_C,Cop_C,Ene_C,MHDV_C,LumOld_C,LumNew_C,CUSMA_C,Total_C
0,2021S05075370001.08,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2021S05070010002.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2021S05075370001.09,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2021S05075370120.02,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2021S05070010006.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
6242,2021S05075591003.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6243,2021S05075591004.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6244,2021S05075800300.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6245,2021S05076020800.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# STEP 11: Processing and adding the data to Choropleth and Centroid GDFs

Add the data on employees (by primary residence) to the GDFs produced in Step 7

In [ ]:
# ============================================================
# CENTROIDS EXPORT - DYNAMIC CATEGORIES
# ============================================================

centroids = cent_gdf.merge(census_bytariffs, on='CTDGUID', how='left')

# Build column list dynamically
centroid_cols = ['CTDGUID']
for tar in tars:
    centroid_cols.extend([f'{tar}_B', f'{tar}_E', f'{tar}_C'])
centroid_cols.append('geometry')

centroids = centroids[centroid_cols]
centroids = centroids.to_crs('EPSG:4326')
centroids.to_file('centroids_ct.geojson', driver='GeoJSON')
centroids.to_csv("centroids_ct.csv", index=False)
centroids

,CTDGUID,Auto_B,Auto_E,Auto_C,Alum_B,Alum_E,Alum_C,Steel_B,Steel_E,Steel_C,...,LumNew_B,LumNew_E,LumNew_C,CUSMA_B,CUSMA_E,CUSMA_C,Total_B,Total_E,Total_C,geometry
0,2021S05070010001.00,0,0,0.0,0,0,0.0,0,0,0.0,...,0,0,0.0,0,0,0.0,0,0,0.0,POINT (-52.70223 47.54410)
1,2021S05070010002.00,0,0,0.0,0,0,0.0,0,0,0.0,...,0,0,0.0,0,0,0.0,0,0,0.0,POINT (-52.73126 47.54049)
2,2021S05070010003.01,0,0,0.0,0,0,0.0,0,0,0.0,...,1,7,0.0,1,7,0.0,1,7,0.0,POINT (-52.74317 47.54068)
3,2021S05070010003.02,0,0,0.0,0,0,0.0,0,0,0.0,...,0,0,0.0,3,67,0.0,3,67,0.0,POINT (-52.75658 47.53958)
4,2021S05070010004.01,2,13,0.0,4,19,0.0,1,2,0.0,...,2,30,0.0,14,177,0.0,14,177,0.0,POINT (-52.77225 47.55991)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6242,2021S05079700100.00,0,0,0.0,0,0,0.0,0,0,0.0,...,1,1,0.0,2,3,0.0,2,3,0.0,POINT (-121.49541 53.98815)
6243,2021S05079700101.00,1,15,0.0,1,15,0.0,2,18,0.0,...,3,119,0.0,12,162,0.0,12,162,0.0,POINT (-122.56527 53.74524)
6244,2021S05079700102.00,0,0,0.0,0,0,0.0,0,0,0.0,...,1,1,0.0,8,15,0.0,8,15,0.0,POINT (-123.00732 53.68513)
6245,2021S05079700103.01,0,0,0.0,0,0,0.0,0,0,0.0,...,0,0,0.0,0,0,0.0,0,0,0.0,POINT (-122.62562 54.01530)


In [27]:
# ============================================================
# PERCENTAGE BY TARIFFS - DYNAMIC CATEGORIES
# ============================================================

perc_bytariffs = census_bytariffs.copy()

for tar in tars:
    tar_3 = f'{tar}_3'

    perc_bytariffs[tar_3] = (
        perc_bytariffs[f'{tar}_C']/perc_bytariffs['Census']
    )

perc_bytariffs = perc_bytariffs[['CTDGUID'] + [f'{tar}_3' for tar in tars]]
perc_bytariffs['CUSMA_3'] = perc_bytariffs['CUSMA_3'].clip(upper=1)
perc_bytariffs['Total_3'] = perc_bytariffs['Total_3'].clip(upper=1)
perc_bytariffs

,CTDGUID,Auto_3,Alum_3,Steel_3,Cop_3,Ene_3,MHDV_3,LumOld_3,LumNew_3,CUSMA_3,Total_3
0,2021S05075370001.08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2021S05070010002.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2021S05075370001.09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2021S05075370120.02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2021S05070010006.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
6242,2021S05075591003.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6243,2021S05075591004.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6244,2021S05075800300.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6245,2021S05076020800.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [28]:
# ============================================================
# CHOROPLETH EXPORT - DYNAMIC CATEGORIES
# ============================================================

choropleth = choro_gdf.merge(perc_bytariffs, on='CTDGUID', how='right')

# Build column list dynamically
choro_export_cols = ['CTDGUID']
for tar in tars:
    choro_export_cols.extend([f'{tar}_1', f'{tar}_2', f'{tar}_3'])
choro_export_cols.append('geometry')

choropleth = choropleth[choro_export_cols]
choropleth = choropleth.to_crs('EPSG:4326')
choropleth

,CTDGUID,Auto_1,Auto_2,Auto_3,Alum_1,Alum_2,Alum_3,Steel_1,Steel_2,Steel_3,...,LumNew_1,LumNew_2,LumNew_3,CUSMA_1,CUSMA_2,CUSMA_3,Total_1,Total_2,Total_3,geometry
0,2021S05075370001.08,0.000000,0.000000,NaN,0.009615,0.005025,NaN,0.000000,0.000000,NaN,...,0.009615,0.003015,NaN,0.019231,0.008040,NaN,0.019231,0.008040,NaN,"POLYGON ((-79.85362 43.19320, -79.85380 43.192..."
1,2021S05070010002.00,0.000000,0.000000,NaN,0.000000,0.000000,NaN,0.000000,0.000000,NaN,...,0.000000,0.000000,NaN,0.000000,0.000000,NaN,0.000000,0.000000,NaN,"POLYGON ((-52.72050 47.55154, -52.71877 47.550..."
2,2021S05075370001.09,0.016129,0.006329,NaN,0.016129,0.006329,NaN,0.016129,0.006329,NaN,...,0.000000,0.000000,NaN,0.016129,0.006329,NaN,0.016129,0.006329,NaN,"POLYGON ((-79.85586 43.18790, -79.85592 43.187..."
3,2021S05075370120.02,0.020528,0.074997,NaN,0.041056,0.081847,NaN,0.032258,0.019487,NaN,...,0.005865,0.016653,NaN,0.143695,0.179284,NaN,0.149560,0.188378,NaN,"POLYGON ((-79.94562 43.16920, -79.94637 43.167..."
4,2021S05070010006.00,0.000000,0.000000,NaN,0.003115,0.000200,NaN,0.000000,0.000000,NaN,...,0.000000,0.000000,NaN,0.012461,0.002496,NaN,0.012461,0.002496,NaN,"POLYGON ((-52.71107 47.56251, -52.71143 47.562..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6242,2021S05075591003.00,0.007389,0.020922,NaN,0.024631,0.042912,NaN,0.027094,0.039176,NaN,...,0.009852,0.002135,NaN,0.147783,0.110376,NaN,0.152709,0.129483,NaN,"POLYGON ((-82.63730 42.14867, -82.63821 42.136..."
6243,2021S05075591004.00,0.009009,0.025566,NaN,0.022523,0.022039,NaN,0.018018,0.046723,NaN,...,0.004505,0.002057,NaN,0.090090,0.095504,NaN,0.103604,0.121951,NaN,"POLYGON ((-82.69417 42.05047, -82.69499 42.035..."
6244,2021S05075800300.00,0.000000,0.000000,NaN,0.000000,0.000000,NaN,0.000000,0.000000,NaN,...,0.083333,0.039474,NaN,0.083333,0.039474,NaN,0.083333,0.039474,NaN,"POLYGON ((-80.41584 46.44983, -80.41636 46.444..."
6245,2021S05076020800.00,0.009091,0.004559,NaN,0.018182,0.018649,NaN,0.018182,0.018649,NaN,...,0.004545,0.001243,NaN,0.118182,0.069208,NaN,0.118182,0.069208,NaN,"POLYGON ((-97.02578 49.59204, -97.02580 49.591..."


In [ ]:
choropleth.to_file('choropleth_ct.geojson', driver='GeoJSON')
choropleth.to_file('choropleth_ct.shp', driver='ESRI Shapefile')

In [ ]:
choropleth.drop(columns="geometry").to_csv("choropleth_ct.csv", index=False)

# CSV Trails

for double-checking purposes

In [ ]:
business_grouped.drop(columns='geometry').to_csv('trail_ct.csv', index=False)
business_filter.drop(columns='geometry').to_csv('trail2_ct.csv', index=False)
business_census.to_csv('trail3_ct.csv', index=False)

In [ ]:
choro_cols.drop(columns='geometry').to_csv('trail4_ct.csv', index=False)

In [ ]:
jobs.to_csv('trail5_ct.csv')
jobs_rate.to_csv('trail6_ct.csv')
adjusted_jobs.to_csv('trail7_ct.csv')